In [95]:
import re
import unicodedata

import pandas as pd

In [96]:
df = pd.read_csv("../data/raw/batdongsan_com_vn.csv")

In [97]:
def extract_numeric(s:str|None) -> float:
    if not isinstance(s,str) or not s:
        return None

    results = re.search(r"^(\d+.?\d*,?\d*)\D?", s)
    if not results:
        return None
    else:
        num_str = results.group(1)
        num_str = num_str.replace(".","")
        num_str = num_str.replace(",",".")
        return float(num_str)
    
def extract_measuring_unit(s:str|None) -> str:
    if not isinstance(s,str) or not s:
        return None

    results = re.search(r"^\d+.?\d*,?\d*\s*(\D*)", s)
    if not results:
        return None
    else:
        return results.group(1)

In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20855 entries, 0 to 20854
Data columns (total 19 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   price              20853 non-null  object 
 1   area               20853 non-null  object 
 2   n_bedrooms         10453 non-null  object 
 3   n_bathrooms        10091 non-null  object 
 4   legal              18290 non-null  object 
 5   interior           9524 non-null   object 
 6   facing_direction   8622 non-null   object 
 7   balcony_direction  4829 non-null   object 
 8   front_width        10823 non-null  object 
 9   front_road_width   10023 non-null  object 
 10  title              20852 non-null  object 
 11  description        20851 non-null  object 
 12  latitude           20846 non-null  float64
 13  longitude          20846 non-null  float64
 14  verified           20855 non-null  int64  
 15  location           20855 non-null  object 
 16  location_details   208

# Extract numerics

Next, we"ll extract numeric values and start inspecting for issues

In [98]:
df["price_val"] = df.price.apply(extract_numeric)
df["price_unit"] = df.price.apply(extract_measuring_unit)
df["area_val"] = df.area.apply(extract_numeric)
df["area_unit"] = df.area.apply(extract_measuring_unit)
df["n_bedrooms"] = df.n_bedrooms.apply(extract_numeric)         # since this overwrite the original, be careful not to run the cell twice
df["n_bathrooms"] = df.n_bathrooms.apply(extract_numeric)
df["front_width_val"] = df.front_width.apply(extract_numeric)
df["front_width_unit"] = df.front_width.apply(extract_measuring_unit)
df["front_road_width_val"] = df.front_road_width.apply(extract_numeric)
df["front_road_width_unit"] = df.front_road_width.apply(extract_measuring_unit)

df[[
    "price_val", "price_unit", "area_val", "area_unit", "n_bedrooms", "n_bathrooms", 
    "front_width_val", "front_width_unit", "front_road_width_val", "front_road_width_unit"
]].describe(include="all")

,price_val,price_unit,area_val,area_unit,n_bedrooms,n_bathrooms,front_width_val,front_width_unit,front_road_width_val,front_road_width_unit
count,18665.000000,18665,20853.000000,20853,10453.000000,10091.000000,10823.000000,10823,10023.000000,10023
unique,NaN,6,NaN,2,NaN,NaN,NaN,1,NaN,1
top,NaN,tỷ,NaN,m²,NaN,NaN,NaN,m,NaN,m
freq,NaN,16250,NaN,20845,NaN,NaN,NaN,10823,NaN,10023
mean,48.201716,NaN,1592.967421,NaN,4.015594,3.744228,37.061121,NaN,14.893263,NaN
std,331.912772,NaN,16022.456873,NaN,7.196456,6.514958,1945.315351,NaN,14.321417,NaN
min,1.000000,NaN,4.800000,NaN,1.000000,1.000000,1.000000,NaN,1.000000,NaN
25%,3.900000,NaN,62.100000,NaN,2.000000,1.000000,5.000000,NaN,7.000000,NaN
50%,9.150000,NaN,100.000000,NaN,3.000000,2.000000,6.000000,NaN,12.000000,NaN
75%,28.000000,NaN,185.000000,NaN,4.000000,4.000000,10.000000,NaN,19.000000,NaN


From this table, there"s 2 noticeable problems:
- The non-null count drops <- "Thỏa thuận" prices got converted None
- 6 unique, non-null `price_unit` -> Price unit inconsistency  

For the first problem, we'll analyze the observations with "Thỏa thuận" prices 
separately and use the rest for the regression model.  
For the second problem, we just need to standardize the units 

# Inconsistent price units

In [7]:
df["price_unit"].unique()

array(['tỷ', 'triệu', None, 'triệu/m²', 'nghìn/m²', 'tỷ/m²', 'nghìn'],
      dtype=object)

In [99]:
CONVERSION_MAP = {
    "tỷ": 1000,
    "tỷ/m²": 1000,
    "triệu": 1,
    "triệu/m²": 1,
    "nghìn/m²": 1e-3, 
    "nghìn": 1e-3   
}
def price_to_mil(row, val_col_label:str, unit_col_label:str):
    value = row[val_col_label]
    unit = row[unit_col_label]
    conversion_factor = CONVERSION_MAP.get(unit)
    if value and unit:
        return value * conversion_factor
    else:
        return None

In [100]:
# convert all to million
df["price_val"] = df.apply(price_to_mil, axis=1, args=("price_val","price_unit"))

# convert price per m2 units to full price
price_per_area = df.price_unit.isin(["tỷ/m²", "triệu/m²", "nghìn/m²"])
df.loc[price_per_area, "price_val"] = df.loc[price_per_area, "price_val"] * df.loc[price_per_area, "area_val"]

df[[
    "price_val", "area_val", "n_bedrooms", "n_bathrooms", 
    "front_width_val", "front_road_width_val",
]].describe()

,price_val,area_val,n_bedrooms,n_bathrooms,front_width_val,front_road_width_val
count,1.866500e+04,20853.000000,10453.000000,10091.000000,10823.000000,10023.000000
mean,2.607902e+04,1592.967421,4.015594,3.744228,37.061121,14.893263
std,3.136131e+05,16022.456873,7.196456,6.514958,1945.315351,14.321417
min,3.950000e-03,4.800000,1.000000,1.000000,1.000000,1.000000
25%,3.400000e+03,62.100000,2.000000,1.000000,5.000000,7.000000
50%,7.600000e+03,100.000000,3.000000,2.000000,6.000000,12.000000
75%,1.929600e+04,185.000000,4.000000,4.000000,10.000000,19.000000
max,2.800000e+07,736000.000000,250.000000,235.000000,200000.000000,500.000000


# Extract city

In [101]:
def extract_city(s:str, to_english:bool=False) -> str:
    if not isinstance(s,str) or not s:
        return None
    
    results = re.search(r'.*,(.*$)', s)
    if not results:
        return None
    
    city_vi = results.group(1).strip()
    city_vi = city_vi.replace('.','')
    if to_english:
        city_en = unicodedata.normalize('NFD', city_vi)
        # Remove combining diacritical marks
        city_en = re.sub(r'[\u0300-\u036f]', '', city_en)
        # Replace special Vietnamese letter "đ" and "Đ"
        city_en = city_en.replace('đ', 'd').replace('Đ', 'D')
        return city_en
    else:
        return city_vi

In [102]:
df['city'] = df.location.apply(extract_city)
df[['city', 'location']].sample(5)

,city,location
801,Hồ Chí Minh,"Quận 7, Hồ Chí Minh"
11155,Quảng Nam,"Điện Bàn, Quảng Nam"
15457,Đà Nẵng,"Ngũ Hành Sơn, Đà Nẵng"
1249,Hà Nội,"Đống Đa, Hà Nội"
6899,Hồ Chí Minh,"Nhà Bè, Hồ Chí Minh"


# Standardize string/categorical columns

In [22]:
df.city.unique()

array(['Hồ Chí Minh', 'Hà Nội', 'Đà Nẵng', 'Khánh Hòa', 'Bình Dương',
       'Đồng Nai', 'Long An', 'Hòa Bình', 'Hưng Yên', 'Bà Rịa Vũng Tàu',
       'Lâm Đồng', 'Hải Phòng', 'Thanh Hóa', 'Quảng Ngãi', 'Hà Nam',
       'Quảng Ninh', 'Thái Nguyên', 'Tiền Giang', 'Tây Ninh',
       'Kiên Giang', 'Vĩnh Phúc', 'Bắc Ninh', 'Bình Phước', 'Bắc Giang',
       'Đắk Lắk', 'Phú Thọ', 'Nghệ An', 'Gia Lai', 'Ninh Thuận',
       'Thừa Thiên Huế', 'Hải Dương', 'Đắk Nông', 'Bình Thuận', 'Cần Thơ',
       'Ninh Bình', 'Quảng Nam', 'Quảng Bình', 'Bình Định', 'Nam Định',
       'Điện Biên', 'Vĩnh Long', 'Phú Yên', 'Lào Cai', 'Thái Bình',
       'Kon Tum', 'Cà Mau', 'Lạng Sơn', 'Hậu Giang', 'Trà Vinh',
       'Tuyên Quang', 'Yên Bái', 'Đồng Tháp', 'Bến Tre', 'Hà Tĩnh',
       'Bắc Kạn', 'Sóc Trăng', 'An Giang', 'Quảng Trị', 'Sơn La'],
      dtype=object)

In [23]:
df.legal.unique()

array(['Sổ hồng đầy đủ', 'Hợp đồng mua bán', 'HDMB', 'Sổ đỏ/ Sổ hồng',
       'Hợp đồng mua bán.', 'Sổ đỏ/ Sổ hồng.', 'HĐMB', 'Sổ đỏ', nan,
       'Sổ hồng', 'Sổ hồng riêng', 'Có sổ.',
       'Sổ hồng đã hoàn công năm 2025', 'Đã có sổ đỏ.', 'Sổ đỏ.',
       'Đã có sổ', 'Sổ hồng (sử dụng chung).', 'Đang chờ sổ',
       'Bùi Xương Trạch, khương đình, Thanh Xuân.',
       'Đã có sổ đỏ - mua bán vi bằng',
       'Sổ đỏ chính chủ, sẵn sàng giao dịch.', 'Sổ hồng.', 'Có sổ',
       'Sổ hồng lâu dài', 'Bìa đỏ chính chủ.', 'Sổ đỏ/ Sổ hồng sdc',
       'Sổ đỏ/ Sổ hồng đất trồng cây', 'Sổ đỏ/ Sổ hồng chung',
       'Đã có sổ riêng', 'Sổ hồng 50 năm.', 'Có sổ( sử dụng chung).',
       'Sổ hồng vĩnh viễn.', 'Sổ đỏ/ Sổ hồng', 'Pháp lý: Sổ cá nhân.',
       'Sổ đỏ 50 năm', 'Mua bán công chứng vi bằng', 'Nhà đã có sổ đỏ.',
       'Có sổ đỏ.', 'Sổ đỏ lâu dài.', 'Sổ đỏ/sổ hồng', 'Đang chờ sổ.',
       'Giầy tờ pháp lý đầy đủ', 'sổ đỏ sang tên', 'Sổ đỏ chính chủ',
       'Hợp đồng mua bán vi bằng', '

In [103]:
def standardize_legal_doc_types(s: str) -> str:
    if not s or not isinstance(s, str) or not s.strip():
        return None
    
    s = s.lower().strip()

    if re.search(r'sổ đỏ', s) or re.search(r'sđ', s) or re.search(r'bìa đỏ',s):
        if re.search(r'sổ hồng', s) or re.search(r'bìa hồng',s): 
            return 'Giấy tờ hợp lệ'
        return 'Sổ đỏ'
    if re.search(r'sổ hồng', s) or re.search(r'bìa hồng', s):
        return 'Sổ hồng'
    if re.search(r'chờ sổ', s):
        return 'Đang chờ sổ'
    if re.search(r'/', s) or re.search(r'sổ', s) or re.search(r'có số', s) or re.search(r'giấy tờ',s) or re.search(r'pháp l[íý]:? đầy đủ|đầy đủ pháp l[íý]|pháp l[íý] chuẩn|pháp l[íý] rõ ràng',s) or re.search(r'bằng khoán',s):
        return 'Giấy tờ hợp lệ'
    if re.search(r'hợp đồng mua bán|hđmb|hdmb|hợp đồng|hd|hđ', s) or re.search(r'vi bằng|vb', s) or re.search(r'văn bản thõa thuận',s):
        return 'Hợp đồng mua bán'
    if re.search(r'giấy tờ giao khoán', s) or re.search(r'giấy phép', s) or re.search(r'trích lục', s):
        return 'Giấy tờ khác'
    return None

In [104]:
df['legal_docs'] = df.legal.apply(standardize_legal_doc_types)
df[['legal_docs', 'legal']].sample(5)

,legal_docs,legal
3876,Giấy tờ hợp lệ,Sổ đỏ/ Sổ hồng.
9794,Giấy tờ hợp lệ,Mua - Nhận ngay lợi nhuận lên đến 7%/năm Vốn b...
16727,Giấy tờ hợp lệ,Sổ đỏ/ Sổ hồng
15238,None,NaN
5943,Giấy tờ hợp lệ,Sổ đỏ/ Sổ hồng


In [91]:
df[['legal_docs', 'legal']].sample(5)

,legal_docs,legal
10090,Giấy tờ hợp lệ,Sổ đỏ/ Sổ hồng
2378,Giấy tờ hợp lệ,Sổ đỏ/ Sổ hồng
16438,None,NaN
15026,Giấy tờ hợp lệ,Sổ đỏ/ Sổ hồng
3631,Giấy tờ hợp lệ,Sổ đỏ/ Sổ hồng


# Select columns for analysis

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20855 entries, 0 to 20854
Data columns (total 29 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   price                  20853 non-null  object 
 1   area                   20853 non-null  object 
 2   n_bedrooms             10453 non-null  float64
 3   n_bathrooms            10091 non-null  float64
 4   legal                  18290 non-null  object 
 5   interior               9524 non-null   object 
 6   facing_direction       8622 non-null   object 
 7   balcony_direction      4829 non-null   object 
 8   front_width            10823 non-null  object 
 9   front_road_width       10023 non-null  object 
 10  title                  20852 non-null  object 
 11  description            20851 non-null  object 
 12  latitude               20846 non-null  float64
 13  longitude              20846 non-null  float64
 14  verified               20855 non-null  int64  
 15  lo

In [110]:
cols_for_analysis = [
    # highly relevant for analysis
    'price_val', 'area_val', 'n_bedrooms', 'n_bathrooms', 'front_width_val', 'front_road_width_val',
    'legal_docs', 'facing_direction', 'balcony_direction', 'property_type', 'city', 'location', # location -> addr

    # might be useful, just not now
    'latitude', 'longitude', 'title', 'description', 'location_details', 'date_of_posting', 'interior', 'verified'
]
df_pruned = df[cols_for_analysis]
df_pruned = df_pruned.rename({
    'price_val': 'price',
    'area_val':'area',
    'front_width_val':'front_width',
    'front_road_width_val':'front_road_width',
    'city':'city/district',
    'location_details':'address_details',
    'location': 'address',
}, axis=1)
df_pruned.sample(3)

,price,area,n_bedrooms,n_bathrooms,front_width,front_road_width,legal_docs,facing_direction,balcony_direction,property_type,city/district,address,latitude,longitude,title,description,address_details,date_of_posting,interior,verified
17246,8100.0,99.0,NaN,NaN,5.2,20.0,Giấy tờ hợp lệ,Bắc,NaN,Đất nền dự án,Hồ Chí Minh,"Quận 9, Hồ Chí Minh",10.799038,106.827740,Bán đất nền đường 20 dự án Centana City Điền P...,"Tọa lạc tại Centana City Điền Phúc Thành, Đườn...","Dự án Centana City Điền Phúc Thành, Đường Trườ...",04/10/2025,NaN,0
18455,5200.0,35.0,3.0,4.0,4.0,NaN,Giấy tờ hợp lệ,Đông - Nam,NaN,Nhà riêng,Hà Nội,"Hoài Đức, Hà Nội",21.048791,105.704305,Siêu phẩm nhà 5 tầng phố Yên Bệ Hoài Đức Hà Nộ...,"Bán nhà phố Yên Bệ Kim Chung Hoài Đức Hà Nội, ...","Đường Yên Bệ, Xã Kim Chung, Hoài Đức, Hà Nội",30/09/2025,NaN,0
13367,6600.0,70.0,NaN,NaN,6.0,5.0,Giấy tờ hợp lệ,NaN,NaN,"Shophouse, nhà phố thương mại",Hà Nội,"Hoàng Mai, Hà Nội",20.958746,105.846133,Shophouse tầng 1 - lô góc - chân đế chung cư G...,Shophouse tầng 1 - lô góc - chân đế chung cư G...,"Green Park Trần Thủ Độ, số 1, Đường Trần Thủ Đ...",02/10/2025,Không nội thất,0


In [111]:
df_pruned.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20855 entries, 0 to 20854
Data columns (total 20 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   price              18665 non-null  float64
 1   area               20853 non-null  float64
 2   n_bedrooms         10453 non-null  float64
 3   n_bathrooms        10091 non-null  float64
 4   front_width        10823 non-null  float64
 5   front_road_width   10023 non-null  float64
 6   legal_docs         18246 non-null  object 
 7   facing_direction   8622 non-null   object 
 8   balcony_direction  4829 non-null   object 
 9   property_type      20855 non-null  object 
 10  city/district      20855 non-null  object 
 11  address            20855 non-null  object 
 12  latitude           20846 non-null  float64
 13  longitude          20846 non-null  float64
 14  title              20852 non-null  object 
 15  description        20851 non-null  object 
 16  address_details    208

In [112]:
df_pruned.to_csv('../data/interim/batdongsan_com_vn(1).csv')